# MeguvizCAC v0.4-DualLoop — A100-SXM4-80GB (`sm_80`) 4-Teacher Joint Distillation & 1.4M+ Corpus

Implements the complete **A100 (`sm_80`) Efficiency Suite** + **4-Teacher Multilingual & World-Knowledge Distillation** over **v0.3-Loop**:
- **Open-Weight Joint Teachers**: Full-BF16 `Qwen/Qwen3.8-27B` (vLLM Prefix Caching, CN social pragmatics) + `thinkingmachines/Inkling-2-Mini` (EN/JA/KO/EU/ZH-EN & 12-domain World Knowledge + 4,000 counterfactual minimal pairs; auto `vllm -> sm_80 GGUF llama_cpp -> BF16 transformers -> OpenAI API`)
- **Homologous Co-Teacher Tribunal (`agy`)**: Stronger `Gemini 4 Neon High` (+ `XHigh` tiebreak) paired with frozen `Gemini 4 Hydrogen XHigh` Rubric v2.1 across disagreement/structured-human/persona/literary/multilingual boundaries
- **Semantic & Subgroup Alignment**: `BAAI/bge-m3` (`1024-d`, auto `intent_dim`) + **12-Group Zero-Sync GPU `TorchGroupDRO`** (including `structured_human_hard` & `multilingual_world_knowledge`)
- **A100 Training Throughput**: `SDPA` attention during training (disabled cleanly before calibration/ONNX export), length-bucketed `bs=1024` + sqrt LR scaling, pre-tokenized pairs, fused `AdamW`, optional `--compile`, zero-PII `@`/URL coverage, and `<= 2.8 ms/row` INT8/W4-W8 ONNX.


## 1. A100 (`sm_80`) GPU Check, Repo Setup & Dependencies


In [ ]:
!nvidia-smi
import os, torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
print(f"Detected GPU: {gpu_name} | cc={cc[0]}.{cc[1]} | VRAM={vram_gb:.1f} GB | bf16={torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")

REPO_DIR = "/content/MeguvizCAC"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jichuo1/MeguvizCAC.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd /content/MeguvizCAC
!git log -1 --oneline
!pip -q install -U onnx onnx_ir ml_dtypes onnxruntime transformers scipy scikit-learn huggingface_hub
!pip -q install -U vllm || echo "vLLM optional install skipped"
if cc < (8, 9):
    print("sm_80 (A100) detected: installing llama-cpp-python CUDA wheel (optional fallback for Inkling NVFP4 GGUF)")
    !pip -q install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 || true


## 2. Stage-1/2/3 Assets, v0.4 1.25M+ Corpus & Optional Drive Bridge


In [ ]:
import os
!mkdir -p /content/data/stage2 /content/data/v03_teachers /content/data/v04 /content/data/co-teacher-gate

# Optional: mount Drive if storing local 1M dump or Neon/Hydrogen gate artifacts
DRIVE_ROOT = "/content/drive/MyDrive/MeguvizV04"
if os.path.isdir("/content/drive"):
    print("Drive already mounted")

!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.1-data/bili-hydrogen-rescored-350387.jsonl.gz" -o /content/data/bili-hydrogen-rescored-350387.jsonl.gz
!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.2-data/stage2_distill_bundle.zip" -o /content/data/stage2_distill_bundle.zip
!unzip -qo /content/data/stage2_distill_bundle.zip -d /content/data/stage2

!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.3.0/meguvizcac-v0.3-loop-2x4-e192-int8.zip" -o /content/data/v03_teachers/e192.zip
!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.3.0/meguvizcac-v0.3-loop-2x4-e128-int8.zip" -o /content/data/v03_teachers/e128.zip
!unzip -qo /content/data/v03_teachers/e192.zip -d /content/data/v03_teachers/e192
!unzip -qo /content/data/v03_teachers/e128.zip -d /content/data/v03_teachers/e128

V04_REAL_URL = "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.4-data/bili-v04-real-1m.jsonl.gz"
V04_SYNTH_URL = "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.4-data/bili-v04-synthetic-250k.jsonl.gz"
if not os.path.isfile("/content/data/v04/bili-v04-real-1m.jsonl.gz"):
    !curl -fL "{V04_REAL_URL}" -o /content/data/v04/bili-v04-real-1m.jsonl.gz || echo "NOTE: Upload/copy bili-v04-real-1m.jsonl.gz into /content/data/v04/"
if not os.path.isfile("/content/data/v04/bili-v04-synthetic-250k.jsonl.gz"):
    !curl -fL "{V04_SYNTH_URL}" -o /content/data/v04/bili-v04-synthetic-250k.jsonl.gz || echo "NOTE: Upload/copy bili-v04-synthetic-250k.jsonl.gz into /content/data/v04/"
!ls -lh /content/data /content/data/v04


## 3. Fast A100 12L Teacher Scoring (`bs=512`/`1024`) + 20k Active Audit Pool


In [ ]:
!python -m meguviz_cac.v04_teacher_score \
  --real-jsonl /content/data/v04/bili-v04-real-1m.jsonl.gz \
  --stage1-data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --qwen-fused /content/data/stage2/stage2-fused-train.jsonl.gz \
  --qwen-distill /content/data/stage2/qwen38_27b_distill.jsonl \
  --synth-path /content/data/v04/bili-v04-synthetic-250k.jsonl.gz \
  --train-12l-teacher --teacher-backbone hfl/chinese-macbert-base --teacher-epochs 2 \
  --active-pool-cap 20000 \
  --out-scores /content/data/v04-teacher-scores.jsonl.gz \
  --active-pool-out /content/data/v04-active-audit-pool.jsonl


## 4. Full-BF16 `Qwen/Qwen3.8-27B` vLLM APC Distillation (`sm_80`) + Counterfactual Chinese Twins


In [ ]:
# On A100-SXM4-80GB (cc=8.0 < 8.9), qwen_distill auto-selects full BF16 Qwen/Qwen3.8-27B + vLLM Prefix Caching
!python -m meguviz_cac.qwen_distill \
  --pool /content/data/v04-active-audit-pool.jsonl \
  --out /content/data/v04-qwen38-bf16-distill.jsonl \
  --twins-out /content/data/v04-qwen38-twins.jsonl \
  --backend auto --comments-per-prompt 8 --gen-twins 800 --max-minutes 28


## 5. `thinkingmachines/Inkling-2-Mini` Multilingual & World-Knowledge Distillation (4k Overlap + 4k Non-ZH + 4k ZH-EN + 4k WK Pairs)


In [ ]:
# Auto backend priority: [--api-base -> vllm -> llama_cpp GGUF (UD-Q8_K_XL on sm_80) -> BF16 transformers]
!python -m meguviz_cac.inkling_distill \
  --real-jsonl /content/data/v04/bili-v04-real-1m.jsonl.gz \
  --stage1-data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --active-pool /content/data/v04-active-audit-pool.jsonl \
  --pool-out /content/data/v04-inkling-pool.jsonl \
  --out-distill /content/data/v04-inkling-distill.jsonl \
  --out-wk-pairs /content/data/v04-inkling-wk-pairs.jsonl \
  --out-wk-corpus /content/data/v04-inkling-wk-corpus.jsonl \
  --select-pool --score-pool --gen-world-knowledge 4000 \
  --non-zh-cap 4000 --mixed-cap 4000 --overlap-cap 4000 \
  --backend auto --max-minutes 35


## 6. Stronger `Gemini 4 Neon` + `Hydrogen` Homologous Co-Teacher Tribunal Bridge

Run the tribunal commands below on your local workstation (where `relabel-neon/tools/agy.exe` is authenticated), then upload `v04-joint-distill-fused.jsonl.gz` and `v04-twins-clean-pairs.jsonl` to `/content/data/` (or Drive):
```bash
python -m meguviz_cac.v04_co_teacher_gate --route --qwen-distill v04-corpus/v04-qwen38-bf16-distill.jsonl --inkling-distill v04-corpus/v04-inkling-distill.jsonl --teacher-scores v04-corpus/v04-teacher-scores.jsonl.gz
python -m meguviz_cac.v04_co_teacher_gate --score --enable-xhigh-tiebreak --neon-batch-size 8 --hydrogen-batch-size 10
python -m meguviz_cac.v04_co_teacher_gate --audit-twins --twin-path v04-corpus/v04-qwen38-twins.jsonl --twin-path v04-corpus/v04-inkling-wk-pairs.jsonl
python -m meguviz_cac.v04_co_teacher_gate --fuse-joint --qwen-distill v04-corpus/v04-qwen38-bf16-distill.jsonl --inkling-distill v04-corpus/v04-inkling-distill.jsonl
```
If Neon/Hydrogen tribunal artifacts are not uploaded yet, the cell below still runs `--fuse-joint` in Colab over `Qwen + Inkling` (Tier 3/4) cleanly.


In [ ]:
import os, shutil
from pathlib import Path

drive_gate = Path("/content/drive/MyDrive/MeguvizV04/co-teacher-gate")
for fname in ("v04-neon-gold.jsonl", "v04-hydrogen-gold.jsonl", "v04-neon-tiebreak.jsonl", "v04-twins-clean-pairs.jsonl"):
    src = drive_gate / fname
    if src.is_file():
        shutil.copy2(src, Path("/content/data/co-teacher-gate") / fname)

if not os.path.isfile("/content/data/v04-joint-distill-fused.jsonl.gz"):
    !python -m meguviz_cac.v04_co_teacher_gate --fuse-joint \
      --gate-dir /content/data/co-teacher-gate \
      --qwen-distill /content/data/v04-qwen38-bf16-distill.jsonl \
      --inkling-distill /content/data/v04-inkling-distill.jsonl \
      --out-fused /content/data/v04-joint-distill-fused.jsonl.gz \
      --audit-out /content/data/v04-four-teacher-audit.json
!cat /content/data/v04-four-teacher-audit.json || true


## 7. Multilingual `BAAI/bge-m3` Intent Cache (`dim=1024`) + 12-GroupDRO Corpus Merge & Dry-Run


In [ ]:
!python -m meguviz_cac.v04_teacher_score \
  --build-intent-cache /content/data/v04_semantic_intent_cache.npz \
  --intent-encoder BAAI/bge-m3 \
  --qwen-distill /content/data/stage2/qwen38_27b_distill.jsonl \
  --qwen-fused /content/data/stage2/stage2-fused-train.jsonl.gz \
  --extra-qwen-distill /content/data/v04-qwen38-bf16-distill.jsonl \
  --extra-qwen-distill /content/data/v04-inkling-distill.jsonl \
  --extra-qwen-distill /content/data/v04-inkling-wk-corpus.jsonl \
  --extra-qwen-distill /content/data/v04-joint-distill-fused.jsonl.gz

!python -m meguviz_cac.v04_corpus_prep \
  --stage1-data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --qwen-distill /content/data/stage2/qwen38_27b_distill.jsonl \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --v04-synth /content/data/v04/bili-v04-synthetic-250k.jsonl.gz \
  --v04-real-jsonl /content/data/v04/bili-v04-real-1m.jsonl.gz \
  --teacher-scores /content/data/v04-teacher-scores.jsonl.gz \
  --v04-qwen-distill /content/data/v04-qwen38-bf16-distill.jsonl \
  --v04-inkling-distill /content/data/v04-inkling-distill.jsonl \
  --v04-joint-distill /content/data/v04-joint-distill-fused.jsonl.gz \
  --v04-multilingual-corpus /content/data/v04-inkling-wk-corpus.jsonl \
  --export-out /content/data/v04-merged-1m4.jsonl.gz

!python -m meguviz_cac.train_v04_dualloop --dry-run \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --extra-pairs stage2-qwen/qwen38_27b_twins.jsonl \
  --extra-pairs /content/data/v04-qwen38-twins.jsonl \
  --extra-pairs /content/data/v04-inkling-wk-pairs.jsonl \
  --extra-pairs /content/data/co-teacher-gate/v04-twins-clean-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --vocab /content/data/v03_teachers/e192/vocab.txt


## 8. 60-Second End-to-End Smoke Test (A100 `bs=1024`, Zero-Sync GPU `TorchGroupDRO`, SDPA -> Export-Safe)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop --smoke-test \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --extra-pairs stage2-qwen/qwen38_27b_twins.jsonl \
  --extra-pairs /content/data/v04-qwen38-twins.jsonl \
  --extra-pairs /content/data/v04-inkling-wk-pairs.jsonl \
  --extra-pairs /content/data/co-teacher-gate/v04-twins-clean-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-smoke \
  --out /content/out_v04_smoke


## 9. Primary Model: `v0.4-dualloop-2x4-e192` (`12-GroupDRO`, `mHC-4`, `LOTUS-4`, `CharStem`, `Fused LN`)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --extra-pairs stage2-qwen/qwen38_27b_twins.jsonl \
  --extra-pairs /content/data/v04-qwen38-twins.jsonl \
  --extra-pairs /content/data/v04-inkling-wk-pairs.jsonl \
  --extra-pairs /content/data/co-teacher-gate/v04-twins-clean-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-dualloop-2x4-e192 \
  --embed-dim 192 --core-layers 2 --num-loops 4 --normal-max-loops 3 \
  --num-latents 4 --mhc-streams 4 --lora-rank 16 --lora-targets qvo_down \
  --epochs 2 --refine-epochs 1 --qat-epochs 2 --batch-size 0 \
  --out /content/out_v04_2x4_e192


## 10. OOD Generalization Ablation: `v0.4-dualloop-2x4-e128` (`E=128`, `mHC-4`, `LOTUS-4`, `CharStem`)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --extra-pairs stage2-qwen/qwen38_27b_twins.jsonl \
  --extra-pairs /content/data/v04-qwen38-twins.jsonl \
  --extra-pairs /content/data/v04-inkling-wk-pairs.jsonl \
  --extra-pairs /content/data/co-teacher-gate/v04-twins-clean-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-dualloop-2x4-e128 \
  --embed-dim 128 --core-layers 2 --num-loops 4 --normal-max-loops 3 \
  --num-latents 4 --mhc-streams 4 --lora-rank 16 --lora-targets qvo_down \
  --epochs 2 --refine-epochs 1 --qat-epochs 2 --batch-size 0 \
  --out /content/out_v04_2x4_e128


## 11. Differential Attention Ablation: `v0.4-dualloop-2x4-e192-diff` (+ Paired DiffAttn)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --extra-pairs stage2-qwen/qwen38_27b_twins.jsonl \
  --extra-pairs /content/data/v04-qwen38-twins.jsonl \
  --extra-pairs /content/data/v04-inkling-wk-pairs.jsonl \
  --extra-pairs /content/data/co-teacher-gate/v04-twins-clean-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-dualloop-2x4-e192-diff \
  --embed-dim 192 --core-layers 2 --num-loops 4 --normal-max-loops 3 \
  --num-latents 4 --mhc-streams 4 --lora-rank 16 --lora-targets qvo_down --diff-attn \
  --epochs 2 --refine-epochs 1 --qat-epochs 2 --batch-size 0 \
  --out /content/out_v04_2x4_e192_diff


## 12. Ultralight Edge Model: `v0.4-dualloop-1x5-e128` (`1x5`, `E=128`, `mHC-1`, `CharStem`)


In [ ]:
!python -m meguviz_cac.train_v04_dualloop \
  --data /content/data/v04-merged-1m4.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --extra-pairs stage2-qwen/qwen38_27b_twins.jsonl \
  --extra-pairs /content/data/v04-qwen38-twins.jsonl \
  --extra-pairs /content/data/v04-inkling-wk-pairs.jsonl \
  --extra-pairs /content/data/co-teacher-gate/v04-twins-clean-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --version v0.4-dualloop-1x5-e128 \
  --embed-dim 128 --core-layers 1 --num-loops 5 --normal-max-loops 4 \
  --num-latents 4 --mhc-streams 1 --lora-rank 16 --lora-targets qvo_down \
  --epochs 2 --refine-epochs 1 --qat-epochs 2 --batch-size 0 \
  --out /content/out_v04_1x5_e128


## 13. Cross-Version Comparison (`v0.1` -> `v0.4`), Multilingual Trap Gates & Bundle Packaging


In [ ]:
import json, zipfile
from pathlib import Path

baselines = [
    ("v0.1 (4L bge-small INT8)", 0.9552, None, None, None, 0.8537, 10.67, 0.8667, None, 23.94, 4.0, 9.73, "N/A", "N/A"),
    ("v0.2 (4L Dual-Teacher INT8)", 0.9530, None, None, None, 0.8598, 10.57, 0.9167, None, 23.94, 4.0, 9.86, "N/A", "N/A"),
    ("v0.3-loop-2x4-e192", 0.9565, 0.9406, None, None, 0.8487, 11.10, 0.9000, 0.7879, 14.71, 1.49, 2.09, "skip", "N/A"),
]
run_dirs = [
    Path("/content/out_v04_2x4_e192"),
    Path("/content/out_v04_2x4_e128"),
    Path("/content/out_v04_2x4_e192_diff"),
    Path("/content/out_v04_1x5_e128"),
]

hdr = (
    f"{'Model':30s} | {'AUROC':6s} | {'PRAUC':6s} | {'R@P95':6s} | {'R@P99':6s} | {'Spear':6s} | "
    f"{'MAE':5s} | {'Probe':6s} | {'StHum':6s} | {'MB':5s} | {'ms32':6s} | {'W4':4s} | {'Mono':4s} | Gates"
)
print(hdr)
print("-" * len(hdr))
def _fmt(v, fmt=".4f", dash="-"):
    return format(v, fmt) if v is not None else dash

for name, ta, pra, rp95, rp99, ts, mae, pa, sh, mb, st, ms, edge, mono in baselines:
    print(
        f"{name:30s} | {_fmt(ta)} | {_fmt(pra)} | {_fmt(rp95)} | {_fmt(rp99)} | {_fmt(ts)} | "
        f"{_fmt(mae, '.2f')} | {_fmt(pa)} | {_fmt(sh)} | {mb:5.2f} | {ms:6.2f} | {edge:4s} | {mono:4s} | base"
    )

for d in run_dirs:
    mf = list(d.glob("metrics-*.json"))
    if not mf:
        continue
    m = json.loads(mf[0].read_text(encoding="utf-8"))
    tb = m["test_bili_gold"]
    prb = m["probe_u001_u017"]
    ox = m["onnx_int8"]
    traps = (m.get("onnx_verify") or {}).get("hard_trap_gates") or {}
    edge_ok = "OK" if (m.get("edge_w4_w8") or {}).get("ok") else "FAIL"
    mono_ok = "OK" if (m.get("loop_monotonicity") or {}).get("non_regressive_normal_last_ge_step0") else "WARN"
    rg = m.get("release_gates") or {}
    gate_str = f"{rg.get('passed_count', 0)}/{rg.get('total_gates', 0)}"
    print(
        f"{m['version']:30s} | "
        f"{_fmt(tb.get('auroc'))} | "
        f"{_fmt(tb.get('pr_auc'))} | "
        f"{_fmt(tb.get('recall_at_p95'))} | "
        f"{_fmt(tb.get('recall_at_p99'))} | "
        f"{_fmt(tb.get('spearman'))} | "
        f"{_fmt(tb.get('mae'), '.2f')} | "
        f"{_fmt(prb.get('auroc'))} | "
        f"{_fmt(traps.get('structured_human_max_prob'))} | "
        f"{ox['total_onnx_bytes'] / (1024**2):5.2f} | "
        f"{ox['compact_batch32_1thread_ms']:6.2f} | "
        f"{edge_ok:4s} | {mono_ok:4s} | {gate_str}"
    )

primary_zip = Path("/content/out_v04_2x4_e192/meguvizcac-v0.4-dualloop-2x4-e192-int8.zip")
if primary_zip.is_file():
    print("\n=== v04_runtime Stress (H01..H09 + M01..M06), Conformal & Gate Verification ===")
    !python -m meguviz_cac.v04_runtime --bundle {primary_zip}

bundle_out = Path("/content/v04_ablation_bundles.zip")
with zipfile.ZipFile(bundle_out, "w", compression=zipfile.ZIP_STORED) as zf:
    for d in run_dirs:
        for pat in ("meguvizcac-*.zip", "metrics-*.json", "manifest.json"):
            for fp in d.glob(pat):
                zf.write(fp, f"{d.name}/{fp.name}")
    for opt_f in (
        Path("/content/data/v04-active-audit-pool.jsonl"),
        Path("/content/data/v04-four-teacher-audit.json"),
    ):
        if opt_f.is_file():
            zf.write(opt_f, opt_f.name)
print(f"Packaged downloadable bundles -> {bundle_out} ({bundle_out.stat().st_size / (1024**2):.2f} MB)")
